# Manuscript-v3 closed-loop article run

This notebook is the canonical, resumable interface to
`scripts/run_article_v3_5000.py`. The production driver, rather than duplicated
notebook code, owns candidate generation and deterministic replacement, two-start
mechanistic acceptance, ridge selection,
untouched-test assessment, scientific admission gates, paired optimization, independent
replay, derivative and physical audits, and Results/Discussion tables.

The full model-function workload is fixed at **5,000 accepted datasets**: 4,000
development inputs and 1,000 untouched test inputs. Rejected mechanistic
candidates remain fully audited but are excluded and deterministically
replaced. It uses ten robustness cases plus the nominal case and a ten-layer
Clarifier. Each route uses one deterministic box-center start per case and
accepts the validated local result without claiming global optimality. The
surrogate route directly runs the seven-variable exact-QP active-set optimizer;
it does not execute the retired embedded-KKT IPOPT problem or any of its seven
gap-continuation stages. The direct smooth-mechanistic route retains its
separate three-stage smoothing continuation. There is no wall-time ceiling in
the full article run. The scientific admission thresholds remain unchanged and still
determine whether results are article-eligible. For this model-function
exercise they are advisory for execution: failures are recorded and propagated
while later stages are attempted without refitting. Non-finite or incomplete
objects needed by a later stage and run-integrity failures remain fatal.

Every projection retains the same strictly convex QP. Its independent dual
audit reconstructs multipliers with deterministic bounded-variable least
squares (BVLS); a failed cold numerical attempt may use the two declared cold
OSQP retry settings, without regularizing or otherwise changing the problem.

The separate, already-completed 500-input preflight record used 400 development
inputs, 100 untouched test inputs, five robustness cases, and five Clarifier
layers. Under the then-current protocol, its limited optimization smoke used
one center start and a 600-second (10-minute) ceiling for each embedded-surrogate
IPOPT continuation stage. That historical solver path and every preflight
artifact are excluded from the revised full article optimization.


## Immutable-source execution

The production contract hashes this notebook byte-for-byte. For a robust
article execution, keep `main_closed_loop.ipynb` unmodified while the run is in
progress. The safest command is to execute it into a different output file:

```powershell
uv run jupyter nbconvert --to notebook --execute main_closed_loop.ipynb `
  --output main_closed_loop.executed.ipynb --ExecutePreprocessor.timeout=-1
```

For interactive work, open a copy of the notebook or disable autosave; do not
save execution counts or outputs back into this source notebook until the run
has finished. All scientific checkpoints live under the selected result
directory, so rerunning a stage resumes verified work rather than starting
over.


In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import asdict
from pathlib import Path

import pandas as pd
from IPython.display import display

from closed_loop.manuscript_v3 import ARTICLE_FULL
from scripts.run_article_v3_5000 import (
    DEFAULT_RUN_ID,
    OPTIMIZATION_PROTOCOL,
    RUN_ID_PATTERN,
    main as run_article,
    resolve_run_directory,
)

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts" / "run_article_v3_5000.py").is_file():
    raise RuntimeError(
        "Run this notebook from the surrogate-optimization-arch repository root."
    )

# Configure with ARTICLE_V3_RUN_ID before starting, or edit this assignment.
RUN_ID = os.environ.get("ARTICLE_V3_RUN_ID", DEFAULT_RUN_ID)
if RUN_ID_PATTERN.fullmatch(RUN_ID) is None or ".." in RUN_ID:
    raise ValueError(
        "ARTICLE_V3_RUN_ID must match article_full_5000_<identifier>."
    )
RUN_ROOT = resolve_run_directory(RUN_ID)

# This explicit flag authorizes only the runner's pinned, one-time migration of
# the already-started article_full_5000_001 optimization contract. Generation,
# fitting, and assessment remain byte-verified and are reused. The runner
# refuses the migration for any other run or predecessor contract.
AUTHORIZE_SINGLE_START_EXACT_QP_MIGRATION = RUN_ID == DEFAULT_RUN_ID

profile = asdict(ARTICLE_FULL)
expected_profile = {
    "name": "article_full",
    "development_count": 4_000,
    "test_count": 1_000,
    "robustness_count": 10,
    "layer_count": 10,
    "article_eligible": True,
    "enforce_admission_gate": True,
}
for field, expected in expected_profile.items():
    if profile.get(field) != expected:
        raise RuntimeError(
            f"ARTICLE_FULL contract mismatch for {field}: "
            f"{profile.get(field)!r} != {expected!r}"
        )
if profile["development_count"] + profile["test_count"] != 5_000:
    raise RuntimeError("The article profile must require exactly 5,000 accepted inputs.")

contract_config = json.loads(
    (ROOT / "config" / "params_manuscript_v3.json").read_text(encoding="utf-8")
)
optimization = contract_config["optimization"]
article_config = contract_config["profiles"]["article_full"]
required_optimization = {
    "protocol": "single_start_exact_qp_active_set",
    "runner_protocol": "single_center_local_exact_qp_v1",
    "surrogate_protocol": "seven_variable_exact_qp_single_start_v1",
    "direct_protocol": "smooth_direct_single_center_v1",
    "start_count": 1,
    "direct_start_count": 1,
    "surrogate_embedded_kkt_ipopt_enabled": False,
    "direct_smoothing_continuation_retained": True,
    "optimization_case_count": 11,
    "nominal_case_count": 1,
    "robustness_case_count": 10,
    "optimization_case_failure_stops_workflow": False,
}
for field, expected in required_optimization.items():
    if optimization.get(field) != expected:
        raise RuntimeError(
            f"Optimization contract mismatch for {field}: "
            f"{optimization.get(field)!r} != {expected!r}"
        )
if article_config.get("optimization_start_count") != 1:
    raise RuntimeError("The article profile requires one start per route and case.")
if len(optimization.get("smooth_sequence", [])) != 3:
    raise RuntimeError("The direct route must retain its three smoothing stages.")
if optimization.get("surrogate_gap_sequence") != []:
    raise RuntimeError("The surrogate route must not execute gap-continuation stages.")
if OPTIMIZATION_PROTOCOL != optimization["runner_protocol"]:
    raise RuntimeError("The runner and configuration optimization protocols differ.")

display(pd.Series({
    "run_id": RUN_ID,
    "run_directory": str(RUN_ROOT),
    "accepted_dataset_target": 5_000,
    "accepted_development_target": profile["development_count"],
    "accepted_untouched_test_target": profile["test_count"],
    "candidate_attempt_count": "reported after generation; may exceed 5,000",
    "candidate_replacement_policy": "audit, exclude, deterministically replace",
    "robustness_cases": profile["robustness_count"],
    "clarifier_layers": profile["layer_count"],
    "starts_per_route_and_case": 1,
    "surrogate_optimization": optimization["surrogate_protocol"],
    "surrogate_embedded_ipopt_stages": 0,
    "direct_optimization": optimization["direct_protocol"],
    "direct_smoothing_stages": len(optimization["smooth_sequence"]),
    "local_optimum_accepted": True,
    "global_optimality_claimed": False,
    "full_run_wall_time_ceiling": None,
    "scientific_admission_gate_enforced_for_article_eligibility": True,
    "admission_gate_execution_policy": "advisory; scientific eligibility unchanged",
}, name="value").to_frame())


## Status and artifact views

The helpers below are read-only. Each stage call publishes atomic checkpoints;
the `finally` block displays the latest state and paths. A scientific gate
failure is visible in those artifacts but does not stop this model-function
exercise. Hard computational and integrity failures still stop the stage.


In [ ]:
STAGE_ARTIFACTS = {
    "generation": (
        "inputs/contract.json",
        "inputs/contract_migrations/article-v3-generation-replacement-v1.json",
        "inputs/contract_migrations/article-v3-projection-audit-v1.json",
        "inputs/contract_migrations/article-v3-direct-active-set-v1.json",
        "datasets/design.npz",
        "datasets/development/all_attempts.csv",
        "datasets/development/accepted_provenance.csv",
        "datasets/development/accepted_inputs.npz",
        "datasets/development/mechanistic_accepted_v3.npz",
        "datasets/development/accepted_diagnostics.csv",
        "datasets/development/base_checkpoint_migration.csv",
        "datasets/development/replacement_summary.json",
        "datasets/development/block_complete.json",
        "datasets/test/all_attempts.csv",
        "datasets/test/accepted_provenance.csv",
        "datasets/test/accepted_inputs.npz",
        "datasets/test/mechanistic_accepted_v3.npz",
        "datasets/test/accepted_diagnostics.csv",
        "datasets/test/base_checkpoint_migration.csv",
        "datasets/test/replacement_summary.json",
        "datasets/test/block_complete.json",
    ),
    "assessment": (
        "models/ridge_complete.json",
        "metrics/assessment_complete.json",
        "metrics/admission_gate.json",
        "metrics/untouched_prediction_metrics.csv",
        "metrics/physical_violations_assessment.csv",
    ),
    "complete": (
        "optimization/optimization_complete.json",
        "metrics/smooth_reference_test_complete.json",
        "metrics/physical_violations_all_analysis.csv",
        "report/tables/report_manifest.json",
    ),
}


def read_json(relative_path: str) -> dict:
    path = RUN_ROOT / relative_path
    if not path.is_file():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def artifact_table(through: str) -> pd.DataFrame:
    ordered = ["run_state.json"]
    for stage in ("generation", "assessment", "complete"):
        ordered.extend(STAGE_ARTIFACTS[stage])
        if stage == through:
            break
    rows = []
    for relative in dict.fromkeys(ordered):
        path = RUN_ROOT / relative
        rows.append({
            "artifact": relative,
            "exists": path.is_file(),
            "bytes": path.stat().st_size if path.is_file() else None,
            "absolute_path": str(path),
        })
    return pd.DataFrame(rows)


def show_status(through: str) -> None:
    state = read_json("run_state.json")
    display(pd.Series(
        state or {"stage": through, "status": "no state published"},
        name="value",
    ).to_frame())
    display(artifact_table(through))


def invoke_stage(through: str) -> None:
    print(f"Invoking production runner through {through!r} for {RUN_ID!r}.")
    try:
        run_article(
            run_id=RUN_ID,
            through=through,
            authorize_single_start_exact_qp_migration=(
                AUTHORIZE_SINGLE_START_EXACT_QP_MIGRATION
            ),
        )
    finally:
        show_status(through)


## 1. Complete the accepted 4,000/1,000 mechanistic blocks

Candidate round 0 retains the independent development/test Latin hypercubes
and resumes their row-level, two-start nonsmooth mechanistic checkpoints. Each
rejected candidate remains in the attempt ledger with all available audits but
is excluded from the accepted dataset. The runner then continues that block's
persisted SplitMix64 state in deterministic row-major supplemental rounds,
each sized to the remaining deficit, until exactly 4,000 development and 1,000
test rows have been accepted. Accepted replacements fill failed original slots
in ascending order; neither candidates nor accepted rows cross block boundaries.

The accepted union is conditioned on mechanistic acceptance and is not one
global Latin hypercube. Generation reports therefore distinguish the attempted
candidate denominator from the accepted-row denominator and retain every
candidate-to-final-slot mapping. The phrase "untouched test" means untouched by
fitting and tuning, not unconditional sampling from the entire input box.

For the already-started default run, the runner verifies the complete migration
chain and preserves the accepted generation, fitted surrogate, and assessment
artifacts. The newest pinned migration changes only the optimization protocol;
it does not regenerate data or refit the surrogate.


In [ ]:
invoke_stage("generation")

generation_rows = []
attempt_status_rows = []
for block, required in (("development", 4_000), ("test", 1_000)):
    attempts_path = RUN_ROOT / "datasets" / block / "all_attempts.csv"
    provenance_path = RUN_ROOT / "datasets" / block / "accepted_provenance.csv"
    attempts = pd.read_csv(attempts_path) if attempts_path.is_file() else pd.DataFrame()
    provenance = (
        pd.read_csv(provenance_path) if provenance_path.is_file() else pd.DataFrame()
    )
    summary = read_json(f"datasets/{block}/replacement_summary.json")
    generation_rows.append({
        "block": block,
        "candidate_attempts": len(attempts),
        "required_accepted_rows": required,
        "accepted_rows": len(provenance),
        "rejected_attempts": max(0, len(attempts) - len(provenance)),
        "base_accepted_rows": summary.get("base_accepted_count"),
        "supplemental_attempts": summary.get("supplemental_attempt_count"),
        "supplemental_rounds": summary.get("supplemental_round_count"),
        "provenance_complete": len(provenance) == required,
    })
    if "attempt_status" in attempts:
        attempt_status_rows.extend(
            {"block": block, "attempt_status": status, "count": int(count)}
            for status, count in attempts["attempt_status"].value_counts(
                dropna=False
            ).items()
        )
display(pd.DataFrame(generation_rows))
display(pd.DataFrame(
    attempt_status_rows,
    columns=("block", "attempt_status", "count"),
))


## 2. Fit and assess on the untouched 1,000-input test block

This call reuses the generation checkpoints, performs the frozen five-fold
ridge selection and trust calibration, opens the untouched block once, and
publishes raw/projected/mechanistic accuracy and physical-violation ledgers.
The gate result remains the scientific article-eligibility decision. A failure
is not waived or refitted; it is recorded, while execution continues because
this run is currently serving as a complete model-function exercise.


In [ ]:
invoke_stage("assessment")

gate = read_json("metrics/admission_gate.json")
if gate:
    display(pd.Series(gate, name="value").to_frame())

prediction_path = RUN_ROOT / "metrics" / "untouched_prediction_metrics.csv"
if prediction_path.is_file():
    display(pd.read_csv(prediction_path))

physical_path = RUN_ROOT / "metrics" / "physical_violations_assessment.csv"
if physical_path.is_file():
    physical = pd.read_csv(physical_path)
    display(physical.groupby("method", dropna=False).agg(
        rows=("method", "size"),
        maximum_mass_violation=("mass_conservation_violation_max", "max"),
        maximum_nonnegativity_violation=("nonnegativity_violation_max", "max"),
        minimum_coordinate=("minimum_coordinate", "min"),
    ))


## 3. Optimize, independently replay, audit, and report

This resumes the completed assessment, including any recorded scientific gate
failure, and runs the nominal plus ten robustness cases. Both routes use one
deterministic box-center start per case and accept the validated local result.
The surrogate route directly uses the seven-variable exact-QP active-set
solver with normalized constraints and no embedded-KKT IPOPT continuation.
The direct route retains only its separate three-stage smoothing continuation.
The driver then performs fixed-input smooth/nonsmooth replay,
derivative and root-reproduction checks, and raw/projected/smooth/reference
mass-conservation and non-negativity accounting before writing all article
tables. A failed or unresolved case remains in the denominator and does not
suppress subsequent cases or downstream audits. There is no 10-minute ceiling
in this phase.


In [ ]:
invoke_stage("complete")

all_physical_path = RUN_ROOT / "metrics" / "physical_violations_all_analysis.csv"
if all_physical_path.is_file():
    all_physical = pd.read_csv(all_physical_path)
    display(all_physical.groupby("method", dropna=False).agg(
        rows=("method", "size"),
        maximum_mass_violation=("mass_conservation_violation_max", "max"),
        mass_violating_rows=(
            "mass_conservation_violation_count", lambda values: int((values > 0).sum())
        ),
        maximum_nonnegativity_violation=("nonnegativity_violation_max", "max"),
        nonnegative_violating_rows=(
            "nonnegativity_violation_count", lambda values: int((values > 0).sum())
        ),
        minimum_coordinate=("minimum_coordinate", "min"),
    ))

report_directory = RUN_ROOT / "report" / "tables"
report_rows = [
    {
        "report_artifact": path.name,
        "bytes": path.stat().st_size,
        "absolute_path": str(path),
    }
    for path in sorted(report_directory.glob("*"))
    if path.is_file()
]
display(pd.DataFrame(
    report_rows,
    columns=("report_artifact", "bytes", "absolute_path"),
))


## Resumption

If execution is interrupted, rerun the setup and helper cells, then rerun the
cell for the desired terminal stage. Calling `complete` is sufficient to
resume every missing prerequisite. The production driver accepts candidate and
accepted-row checkpoints only when their source, profile, input, stream-state,
and provenance bindings match. Its pinned migration chain preserves the
generation-replacement and projection-audit history, verifies all reusable
generation, fit, and assessment artifacts, and starts the revised optimization
without accepting a partial result from the retired protocol. Any other
mismatch is a run-integrity failure and is not hidden by regeneration.
